In [ ]:
from google.colab import files
uploaded=files.upload()

Saving appointments.csv to appointments.csv


In [ ]:
#1 TASK 1.1

import csv
import json

def parse_row(row):
  converted=row.copy()
  converted["age"]=int(converted["age"])
  converted["fee_inr"]=int(converted["fee_inr"])
  converted["prior_no_shows"]=int(converted["prior_no_shows"])
  converted["distance_km"]=float(converted["distance_km"])
  return converted

Accepted=[]
Rejected=[]
count=0

with open("/content/appointments.csv",newline="") as file:
  reader=csv.DictReader(file)
  print("header:",reader.fieldnames)
  for row in reader:
    count+=1
    try:
      Accepted.append(parse_row(row))
    except ValueError:
      Rejected.append(row)
print("row count:",count)
print("Accpted:",len(Accepted),"Rejected:",len(Rejected))

with open("rejected_rows.json","w") as file:
  json.dump(Rejected,file,indent=2)





header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
row count: 1020
Accpted: 988 Rejected: 32


In [ ]:
# TASK 1.2

import pandas as pd

df=pd.read_csv("/content/appointments.csv")
print("shape:",df.shape)
df.info()
print(df.isnull().sum())
print("duplicates:",df.duplicated().sum())

#1 exact dupliactes rows
df=df.drop_duplicates().copy()
print("drop duplicates:",len(df))

#2 maintain same spelling in boooking channel
df["booking_channel"]=df["booking_channel"].str.strip().str.title()
print("clean channel:",len(df))

#3 real dates
df["appt_date"]=pd.to_datetime(df["appt_date"],format="%d-%m-%Y", errors="coerce")
df["booked_on"]=pd.to_datetime(df["booked_on"],format="%d-%m-%Y")
print("dates:",len(df),"| appt_date NaT:",df["appt_date"].isnull().sum())

#4 drop rows whose appt_date or status is missing
df=df.dropna(subset=["appt_date","status"]).copy()
print("undropped:",len(df))

shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
appointment_id      0
patient_id          0
clinic_id           0
booked_o

In [ ]:
# Task 1.3

import numpy as np

#1 impossible ages
bad_age=(df["age"]<1)|(df["age"]>100)
print("impossible age:",bad_age.sum())
df.loc[bad_age,"age"]=np.nan

#2 fill missing with the median
df["age"]=df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"]=df["distance_km"].fillna(df["distance_km"].median())

#3 iqr---cap----clip
for col in ["distance_km","fee_inr"]:
  q1=df[col].quantile(0.25)
  q3=df[col].quantile(0.75)
  iqr=q3-q1
  lower=q1-1.5*iqr
  upper=q3+1.5*iqr
  capped=((df[col]<lower)|(df[col]>upper)).sum()
  df[col]=np.clip(df[col],lower,upper)
  print(col,"fences:",round(lower,2),round(upper,2),"|capped:",capped)

#4 rounding
df["distance_km"]=df["distance_km"].round(1)
df["fee_inr"]=df["fee_inr"].round().astype(int)
print(df[["age","distance_km","fee_inr"]].isnull().sum())
print(df[["age","distance_km","fee_inr"]].dtypes)

impossible age: 5
distance_km fences: -7.94 25.56 |capped: 26
fee_inr fences: 68.75 1218.75 |capped: 3
age            0
distance_km    0
fee_inr        0
dtype: int64
age              int64
distance_km    float64
fee_inr          int64
dtype: object


In [ ]:
# task 1.4

import numpy as np
df["lead_days"]=(df["appt_date"]-df["booked_on"]).dt.days
df["no_show"]=(df["status"]=="No-Show").astype(int)
print("lead days range:",df["lead_days"].min(),"to",df["lead_days"].max())
print("overall no show rate: %.1f%%"%(df["no_show"].mean()*100))

x=df[["age","distance_km","fee_inr","lead_days"]].to_numpy()
print("means:",np.mean(x,axis=0).round(2))

# booked more than 14 days ahead
mask=df["lead_days"]>14
print("lead days more than 14: cout=",mask.sum(),"share=%.1f%%"%(mask.mean()*100))


#z-scores of distance_km
x=df["distance_km"].to_numpy()
z=(x-np.mean(x))/np.std(x)
print("rows with |z|>2:",(np.abs(z)>2).sum())


lead days range: 1 to 45
overall no show rate: 22.0%
means: [ 48.17   9.5  635.71  11.82]
lead days more than 14: cout= 296 share=29.9%
rows with |z|>2: 53


In [ ]:
# TASK 1.5
import pandas as pd
import requests

clinics=pd.read_csv("/content/clinics.csv")
url="https://api.open-meteo.com/v1/forecast"
temps=[]

for _, clinic in clinics.iterrows():
  temp_c=None
  reson=None

  for attempt in range(2):
    try:
      response=requests.get(url,params={"latitude":clinic["latitude"],"longitude":clinic["longitude"],"hourly":"temperature_2m","forecast_days":1,},timeout=10)

      if response.status_code!=200:
        reason=f"HTTP status {response.status_code}"
        continue
      values=response.json().get("hourly",{}).get("temperature_2m")

      if not values:
        reason="hourly temperature data is missing "
        continue

      temp_c=values[0]
      reason=None if temp_c is not None else "temperature value is empty"
      break

    except requests.exceptions.RequestException as error:
      reason=str(error)
  if temp_c is None:
    print("failed:",clinic["clinic_name"],"-",reason)
  temps.append(temp_c)

clinics["temp_c"]=temps
clinics.to_csv("clinics_enriched.csv",index=False)
print(clinics[["clinic_name","temp_c"]])
print("temperatures found:",clinics["temp_c"].notna().sum(),"of",len(clinics))





                clinic_name  temp_c
0     Kothrud Family Clinic    16.2
1      Hadapsar Care Centre    17.9
2    Sitabuldi Health Point    20.1
3      Malviya Nagar Clinic    15.5
4     Vaishali Nagar Clinic    15.5
5  Edappally Medical Centre    25.3
6        Vijay Nagar Clinic    18.1
7    Gomti Nagar Care Point    17.5
temperatures found: 8 of 8


In [ ]:
# task 1.6
import pandas as pd
columns=["appointment_id","patient_id","clinic_id","booked_on","appt_date","age","gender","distance_km","fee_inr","booking_channel","visit_type","reminder_sent","prior_no_shows","status","lead_days","no_show"]
out=df.sort_values("appointment_id").copy()
out["booked_on"]=out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"]=out["appt_date"].dt.strftime("%Y-%m-%d")
out=out[columns]

print("appointment_id unique:",out["appointment_id"].is_unique)
print("missing values in the 16 columns:",out.isnull().sum().sum())
out.to_csv("appointments_clean.csv",index=False)
print("saved appointments_clean.csv:",out.shape)

def sql_values(v):
  if pd.isna(v):
    return "NULL"
  if isinstance(v,str):
    return "'"+v.replace("'","''") +"'"
  return str(v)
clinics_final=pd.read_csv("clinics_enriched.csv")

with open("load_data.sql","w") as file:
  for _,c in clinics_final.iterrows():
    values=",".join(sql_values(v) for v in c)
    file.write(f"INSERT INTO clinics VALUES({values});\n")

  for _,row in out.iterrows():
    values=",".join(sql_values(v) for v in row)
    file.write(f"INSERT INTO appointments VALUES({values});\n")
print("load_sql.csv written")

with open("load_data.sql") as file:
  for i ,line in enumerate(file):
    if i in (0,8):
      print(line.strip())
    if i>8:
      break







appointment_id unique: True
missing values in the 16 columns: 0
saved appointments_clean.csv: (990, 16)
load_sql.csv written
INSERT INTO clinics VALUES('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,16.2);
INSERT INTO appointments VALUES('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);
